# Tier 2 — Rozproszony system przechowywania artefaktów na Ray

**Wersja distributed** — uruchamiana w klastrze Ray rozproszonym przez Docker Compose
(1 head + N workers + Jupyter w osobnych kontenerach). Notebook łączy się z klastrem
przez Ray Client (`ray://ray-head:10001`), a aktorzy są dystrybuowani po fizycznych
węzłach automatycznie przez scheduler Ray.

Implementacja prostego, HDFS-podobnego systemu plików zbudowanego z aktorów Ray.

### Architektura

```
                 ┌─────────────────────────────┐
                 │         NameNode            │  (named actor: "NameNode")
                 │  - metadane artefaktów      │
                 │  - placement planner        │
                 │  - heartbeat / re-replikacja│
                 └──────────────┬──────────────┘
                                │ (metadane, plan placementu)
                                │
                  ┌─────────────┴─────────────┐
                  │                           │
             ┌────▼────┐  ┌────▼────┐  ┌────▼────┐  ┌────▼────┐
             │DataNode1│  │DataNode2│  │DataNode3│  │DataNode4│   …
             │ chunks  │  │ chunks  │  │ chunks  │  │ chunks  │
             └─────────┘  └─────────┘  └─────────┘  └─────────┘
                  ▲                           ▲
                  │   bezpośredni zapis       │
                  └──── Client ──────────────-┘
```

### Założenia projektowe (zgodne z treścią Tier 2)

- **Artefakt** = `(name, content: str)`. Content dzielony na **chunki po 32 znaki**.
- **Replication factor = 3** — każdy chunk żyje na 3 różnych DataNodes.
- **NameNode** trzyma tylko metadane (`artifact_name → [(chunk_id, hash, [nodes])]`).
  DataNodes **nie rozmawiają między sobą** w trakcie uzgadniania — NameNode jest jedynym koordynatorem.
- **Dane** są wysyłane przez klienta **bezpośrednio na DataNodes** (NameNode tylko zwraca plan).
- **Awarie**: DataNode może być "zabity" (status=DOWN). NameNode wykrywa to przez heartbeat
  i re-replikuje brakujące chunki z żywych kopii na inne żywe węzły.
- **Update inkrementalny**: porównujemy hash chunków per-pozycja — podmieniamy tylko te, które się zmieniły
  (zmiana jednego znaku ≠ re-upload całości).
- **Spójność**: po awarii / re-replikacji wszystkie kopie chunku są bit-identyczne.

### Elementy Ray Actor API spoza standardowych laboratoriów

1. **AsyncIO actors** — `async def` w metodach aktorów + `max_concurrency`, dzięki czemu
   NameNode może równolegle pingować wszystkie DataNodes i równolegle wysyłać chunki w replikacji.
2. **Named actors + namespace** — NameNode rejestrowany jako `name="NameNode", namespace="dfs"`,
   dzięki czemu klient dostaje uchwyt przez `ray.get_actor()` bez przekazywania referencji.
3. **`max_restarts` + `max_task_retries`** — DataNodes mają konfigurowalne ponowne uruchamianie,
   co pokazuje wbudowaną tolerancję awarii Ray (nasza warstwa logiczna i tak ma własną),
   oraz `lifetime="detached"` dla NameNode.

> Referencja: <https://docs.ray.io/en/latest/ray-core/actors.html>

## 1. Setup — połączenie z rozproszonym klastrem Ray

Kontener `jupyter` ma ustawioną zmienną środowiskową `RAY_ADDRESS=ray://ray-head:10001`,
więc `ray.init()` bez argumentów automatycznie łączy się z head node przez Ray Client.

Po połączeniu sprawdzamy listę węzłów klastra (`ray.nodes()`) — powinniśmy widzieć
1 head + N worker nodes.

In [ ]:
import ray
import asyncio
import hashlib
import os
import random
import time
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional

# Idempotentny restart Ray na potrzeby ponownego uruchamiania komórek
if ray.is_initialized():
    ray.shutdown()

# RAY_ADDRESS=ray://ray-head:10001 ustawia docker-compose w kontenerze jupyter.
# Jeśli zmiennej nie ma (uruchomienie lokalne), wpadamy w tryb lokalny.
ray_addr = os.environ.get("RAY_ADDRESS")
ctx = ray.init(
    address=ray_addr,           # None , wtedy lokalny single-node klaster
    namespace="dfs",            # potrzebne dla named actors
    ignore_reinit_error=True,
    logging_level="WARNING",
)

# Pokaż info o klastrze, żeby było widać, że to faktycznie multi-node
dash = ctx.dashboard_url if hasattr(ctx, 'dashboard_url') else None
print(f"Connected to:    {ray_addr or '(local)'}")
print(f"Ray dashboard:   {dash or 'http://localhost:8265'}")
print(f"Cluster nodes:   {len(ray.nodes())}")
print(f"Total CPU cores: {ray.cluster_resources().get('CPU', 0)}")
print("")
print("Per-node breakdown:")
for n in ray.nodes():
    role = "HEAD" if n.get("IsHeadNode") or n.get("Resources", {}).get("node:__internal_head__") else "worker"
    print(f"  {role:6s}  {n.get('NodeManagerHostname', '?'):40s}  "
          f"alive={n.get('Alive')}  cpus={n.get('Resources', {}).get('CPU', 0)}")

SIGTERM handler is not set because current thread is not the main thread.


Connected to:    ray://ray-head:10001
Ray dashboard:   172.24.0.2:8265
Cluster nodes:   3
Total CPU cores: 5.0

Per-node breakdown:
  HEAD    75fd0a298c58                              alive=True  cpus=1.0
  worker  699d1c9a7f9c                              alive=True  cpus=2.0
  worker  4385099a771b                              alive=True  cpus=2.0


## 2. Stałe i helpery

- `CHUNK_SIZE = 32` — wymuszamy dużą liczbę chunków nawet przy krótkich artefaktach,
  żeby demo było czytelne i żeby update inkrementalny miał sens.
- `REPLICATION_FACTOR = 3` — klasyka HDFS.
- `HEARTBEAT_INTERVAL = 1.0 s` — częstotliwość pingu DataNodes przez NameNode.
- `HEARTBEAT_TIMEOUT = 2.0 s` — jeśli ping nie wróci w tym czasie, węzeł uznajemy za DOWN.

In [25]:
CHUNK_SIZE = 32
REPLICATION_FACTOR = 3
HEARTBEAT_INTERVAL = 1.0
HEARTBEAT_TIMEOUT = 2.0   # tolerancyjny


def chunk_string(text: str, size: int = CHUNK_SIZE) -> list[str]:
    """Tnie napis na chunki o stałym rozmiarze. Ostatni chunk może być krótszy."""
    if not text:
        return [""]
    return [text[i:i + size] for i in range(0, len(text), size)]


def chunk_hash(payload: str) -> str:
    """Krótki, deterministyczny fingerprint chunka — używany do detekcji zmian przy update."""
    return hashlib.md5(payload.encode("utf-8")).hexdigest()[:12]

## 3. `DataNode` — aktor przechowujący chunki

Każdy DataNode trzyma chunki w słowniku `chunk_id → payload`. ID chunka to
`{artifact}__{index}__{hash}` — łączy nazwę artefaktu, pozycję i fingerprint zawartości.
Dzięki temu te same dane na różnych pozycjach to różne chunki, a zmiana zawartości
zmienia ID.

Klasa jest **AsyncIO actorem** (`max_concurrency > 1`, metody `async`), żeby NameNode
i klient mogli wysyłać do wielu DataNodes równolegle.

Status `DOWN` jest emulowany flagą — kiedy węzeł jest zabity, każda metoda rzuca
wyjątek (`RayActorError` z perspektywy wołającego) i NameNode to wykrywa.

In [26]:
@ray.remote(max_restarts=2, max_task_retries=1)
class DataNode:
    """AsyncIO actor przechowujący chunki w pamięci.

    Każda metoda jest async, więc Ray uruchamia aktora w trybie AsyncIO
    z max_concurrency = 1000 domyślnie. max_restarts pozwoli Rayowi
    odtworzyć aktora jeśli proces padnie (np. OOM) — naszą logiczną awarię
    (status DOWN) symulujemy jednak ręcznie, żeby kontrolować scenariusz demo.
    """

    def __init__(self, node_id: str):
        self.node_id = node_id
        self.alive = True
        self.chunks: dict[str, str] = {}   # chunk_id -> payload

    # ---- operacje na chunkach
    async def store(self, chunk_id: str, payload: str) -> bool:
        if not self.alive:
            raise RuntimeError(f"DataNode {self.node_id} is DOWN")
        self.chunks[chunk_id] = payload
        return True

    async def fetch(self, chunk_id: str) -> str:
        if not self.alive:
            raise RuntimeError(f"DataNode {self.node_id} is DOWN")
        if chunk_id not in self.chunks:
            raise KeyError(f"Chunk {chunk_id} not on node {self.node_id}")
        return self.chunks[chunk_id]

    async def remove(self, chunk_id: str) -> bool:
        if not self.alive:
            raise RuntimeError(f"DataNode {self.node_id} is DOWN")
        self.chunks.pop(chunk_id, None)
        return True

    async def has_chunk(self, chunk_id: str) -> bool:
        if not self.alive:
            raise RuntimeError(f"DataNode {self.node_id} is DOWN")
        return chunk_id in self.chunks

    # ---- diagnostyka 
    async def ping(self) -> str:
        if not self.alive:
            raise RuntimeError(f"DataNode {self.node_id} is DOWN")
        return "OK"

    async def list_chunks(self) -> dict:
        """Zwraca stan węzła nawet jeśli jest DOWN — dla wglądu administracyjnego."""
        return {
            "node_id": self.node_id,
            "status": "READY" if self.alive else "DOWN",
            "chunk_count": len(self.chunks),
            "chunks": sorted(self.chunks.keys()),
        }

    # ---- sterowanie demo
    async def kill(self):
        """Symulacja awarii — od teraz wszystkie operacje będą rzucać."""
        self.alive = False

    async def revive(self):
        """Reaktywacja węzła (z czystym dyskiem — odzwierciedla wymianę maszyny)."""
        self.alive = True
        self.chunks.clear()

## 4. `NameNode` — koordynator metadanych

NameNode trzyma:

- **`nodes`** — mapę `node_id → DataNode handle`, plus status (`READY`/`DOWN`).
- **`artifacts`** — mapę `artifact_name → list[ChunkMeta]`, gdzie `ChunkMeta` to
  `(index, chunk_id, hash, [locations])`. **Kolejność** chunków jest istotna — to ona
  pozwala zrekonstruować artefakt.

Najważniejsze metody:

- `plan_placement(artifact, num_chunks)` — wybiera dla każdego chunka `R=3` różnych żywych
  węzłów (rotation, żeby rozłożyć obciążenie). Klient potem **sam** zapisuje chunki.
- `commit_chunks(artifact, planned, hashes)` — po sukcesie zapisu klient potwierdza
  metadane.
- `update_artifact(...)` — porównuje hash per-pozycja i zwraca plan dla **tylko zmienionych**
  pozycji. Niezmienione chunki zostają w spokoju.
- `heartbeat_loop()` — background task uruchamiany w `__init__`; pinguje DataNodes,
  oznacza martwe jako DOWN, triggeruje re-replikację brakujących chunków.

Jest to **AsyncIO actor** z `lifetime="detached"` i `name="NameNode"`, namespace `dfs` —
tzn. żyje niezależnie od skryptu, który go stworzył, i klient łapie go przez `get_actor()`.

In [27]:
@dataclass
class ChunkMeta:
    index: int
    chunk_id: str
    hash: str
    locations: list[str] = field(default_factory=list)

    def to_dict(self):
        return {
            "index": self.index,
            "chunk_id": self.chunk_id,
            "hash": self.hash,
            "locations": list(self.locations),
        }

In [28]:
@ray.remote(max_restarts=1)
class NameNode:
    """AsyncIO actor — koordynator metadanych i replikacji."""

    def __init__(self, replication_factor: int = REPLICATION_FACTOR):
        self.replication_factor = replication_factor
        self.nodes: dict[str, dict] = {}             # node_id -> {handle, status}
        self.artifacts: dict[str, list[ChunkMeta]] = {}
        self._round_robin = 0
        self._heartbeat_task: Optional[asyncio.Task] = None
        self._log: list[str] = []

    # ---- rejestracja węzłów ------------------------------------------------
    async def register_node(self, node_id: str, handle):
        self.nodes[node_id] = {"handle": handle, "status": "READY"}
        self._log.append(f"[register] {node_id}")

    async def start_monitoring(self):
        """Startuje pętlę heartbeat (uruchamiana raz, po rejestracji węzłów)."""
        if self._heartbeat_task is None or self._heartbeat_task.done():
            self._heartbeat_task = asyncio.create_task(self._heartbeat_loop())
            self._log.append("[monitor] heartbeat loop started")

    async def stop_monitoring(self):
        if self._heartbeat_task and not self._heartbeat_task.done():
            self._heartbeat_task.cancel()
            self._log.append("[monitor] heartbeat loop stopped")

    # ---- placement planner -------------------------------------------------
    def _alive_nodes(self) -> list[str]:
        return [nid for nid, info in self.nodes.items() if info["status"] == "READY"]

    def _pick_nodes(self, k: int, exclude: set[str] = None) -> list[str]:
        """Wybiera k różnych żywych węzłów techniką round-robin + losowanie."""
        exclude = exclude or set()
        candidates = [n for n in self._alive_nodes() if n not in exclude]
        if len(candidates) < k:
            raise RuntimeError(
                f"Insufficient live DataNodes: need {k}, have {len(candidates)}"
            )
        # Round-robin start point, żeby spread był równomierny
        start = self._round_robin % len(candidates)
        self._round_robin += 1
        rotated = candidates[start:] + candidates[:start]
        return rotated[:k]

    async def plan_placement(
        self, artifact: str, num_chunks: int
    ) -> list[list[str]]:
        """Zwraca listę: dla każdego chunka — listę node_ids gdzie ma trafić.
        Klient potem pisze bezpośrednio do tych DataNodes."""
        return [
            self._pick_nodes(self.replication_factor) for _ in range(num_chunks)
        ]

    async def commit_artifact(
        self, artifact: str, chunk_specs: list[dict]
    ) -> bool:
        """chunk_specs = [{index, chunk_id, hash, locations}, ...] — wywołane PO udanym zapisie."""
        self.artifacts[artifact] = [
            ChunkMeta(
                index=s["index"],
                chunk_id=s["chunk_id"],
                hash=s["hash"],
                locations=list(s["locations"]),
            )
            for s in sorted(chunk_specs, key=lambda x: x["index"])
        ]
        self._log.append(
            f"[commit] {artifact}: {len(chunk_specs)} chunks"
        )
        return True

    # ---- query API ---------------------------------------------------------
    async def get_artifact_metadata(self, artifact: str) -> list[dict]:
        if artifact not in self.artifacts:
            raise KeyError(f"Artifact {artifact!r} not found")
        return [c.to_dict() for c in self.artifacts[artifact]]

    async def list_artifacts(self) -> list[str]:
        return sorted(self.artifacts.keys())

    async def get_node_handle(self, node_id: str):
        if node_id not in self.nodes:
            raise KeyError(f"Unknown node {node_id!r}")
        return self.nodes[node_id]["handle"]

    async def cluster_status(self) -> dict:
        """Pełen przegląd: węzły, ich status, artefakty, mapowanie chunk→nodes."""
        # Zbierz info z DataNodes równolegle. list_chunks nie rzuca dla DOWN — bezpieczne.
        node_infos = await asyncio.gather(
            *[info["handle"].list_chunks.remote() for info in self.nodes.values()],
            return_exceptions=True,
        )
        node_report = {}
        for (nid, info), result in zip(self.nodes.items(), node_infos):
            if isinstance(result, Exception):
                node_report[nid] = {"status": "UNREACHABLE", "error": str(result)}
            else:
                # NameNode-side status (DOWN po heartbeacie) bierzemy z self.nodes,
                # bo na zabitym węźle list_chunks zwraca status=DOWN ale my chcemy też pokazać
                # opinię NameNode (mogą się różnić w tranzytkach przed re-replikacją).
                node_report[nid] = {
                    **result,
                    "namenode_view": info["status"],
                }
        return {
            "replication_factor": self.replication_factor,
            "nodes": node_report,
            "artifacts": {
                name: [c.to_dict() for c in metas]
                for name, metas in self.artifacts.items()
            },
        }

    async def get_log(self) -> list[str]:
        return list(self._log)

    # ---- update + delete ---------------------------------------------------
    async def plan_update(
        self, artifact: str, new_chunks: list[tuple[str, str]]
    ) -> dict:
        """new_chunks = [(payload, hash), ...] w kolejności.

        Zwraca:
          - `unchanged`: lista indeksów do zachowania
          - `to_write`: lista (index, payload, hash, [target_nodes]) — TYLKO zmienione
          - `to_delete`: lista (chunk_id, [nodes]) — chunki które wypadły z artefaktu
            (np. nowa wersja jest krótsza)
        """
        old_metas = self.artifacts.get(artifact, [])
        old_by_index = {m.index: m for m in old_metas}

        unchanged = []
        to_write = []
        for idx, (payload, h) in enumerate(new_chunks):
            if idx in old_by_index and old_by_index[idx].hash == h:
                unchanged.append(idx)
            else:
                targets = self._pick_nodes(self.replication_factor)
                to_write.append({
                    "index": idx,
                    "payload": payload,
                    "hash": h,
                    "targets": targets,
                })

        to_delete = []
        new_len = len(new_chunks)
        for idx, meta in old_by_index.items():
            if idx >= new_len:
                to_delete.append({
                    "chunk_id": meta.chunk_id,
                    "locations": list(meta.locations),
                })

        self._log.append(
            f"[update-plan] {artifact}: unchanged={len(unchanged)} "
            f"rewrite={len(to_write)} drop={len(to_delete)}"
        )
        return {"unchanged": unchanged, "to_write": to_write, "to_delete": to_delete}

    async def commit_update(
        self,
        artifact: str,
        new_metas: list[dict],
    ) -> bool:
        """new_metas: pełna nowa lista chunków artefaktu (po zastosowaniu update)."""
        self.artifacts[artifact] = [
            ChunkMeta(
                index=s["index"],
                chunk_id=s["chunk_id"],
                hash=s["hash"],
                locations=list(s["locations"]),
            )
            for s in sorted(new_metas, key=lambda x: x["index"])
        ]
        self._log.append(f"[update] {artifact}: now {len(new_metas)} chunks")
        return True

    async def delete_artifact(self, artifact: str) -> dict:
        """Usuwa artefakt z metadanych + fizycznie z DataNodes. Zwraca raport."""
        if artifact not in self.artifacts:
            raise KeyError(f"Artifact {artifact!r} not found")

        metas = self.artifacts.pop(artifact)
        # Sprzątamy chunki ze wszystkich (żywych) lokacji
        tasks = []
        for m in metas:
            for nid in m.locations:
                if self.nodes[nid]["status"] == "READY":
                    tasks.append(
                        self.nodes[nid]["handle"].remove.remote(m.chunk_id)
                    )
        results = await asyncio.gather(*tasks, return_exceptions=True)
        errors = [str(r) for r in results if isinstance(r, Exception)]
        self._log.append(f"[delete] {artifact}: removed {len(metas)} chunk(s)")
        return {"deleted_chunks": len(metas), "errors": errors}

    # ---- heartbeat & re-replikacja ----------------------------------------
    async def _heartbeat_loop(self):
        while True:
            try:
                await asyncio.sleep(HEARTBEAT_INTERVAL)
                await self._check_health()
            except asyncio.CancelledError:
                break
            except Exception as e:
                self._log.append(f"[heartbeat-error] {e}")

    async def _check_health(self):
        """Ping wszystkich węzłów równolegle. Zmiana statusu → re-replikacja."""
        node_ids = list(self.nodes.keys())
        if not node_ids:
            return
        ping_refs = [self.nodes[nid]["handle"].ping.remote() for nid in node_ids]

        # ray.wait nie ma timeoutu z async — używamy asyncio.wait_for per-ping
        results = await asyncio.gather(
            *[self._ping_one(nid, ref) for nid, ref in zip(node_ids, ping_refs)],
            return_exceptions=True,
        )

        newly_down = []
        for nid, ok in zip(node_ids, results):
            prev = self.nodes[nid]["status"]
            new = "READY" if ok is True else "DOWN"
            if new != prev:
                self.nodes[nid]["status"] = new
                self._log.append(f"[health] {nid}: {prev} -> {new}")
                if new == "DOWN":
                    newly_down.append(nid)

        if newly_down:
            await self._re_replicate(newly_down)

    async def _ping_one(self, nid: str, ref) -> bool:
        try:
            # Konwersja ObjectRef -> asyncio przez asyncio.wrap_future nie działa wprost;
            # Ray udostępnia await na ObjectRef bezpośrednio w AsyncIO actorach.
            await asyncio.wait_for(ref, timeout=HEARTBEAT_TIMEOUT)
            return True
        except Exception:
            return False

    async def _re_replicate(self, dead_nodes: list[str]):
        """Dla każdego chunka który stracił replikę: skopiuj z żywej kopii na nowy żywy węzeł."""
        dead = set(dead_nodes)
        for artifact, metas in self.artifacts.items():
            for meta in metas:
                lost = [n for n in meta.locations if n in dead]
                if not lost:
                    continue
                surviving = [n for n in meta.locations if n not in dead]
                if not surviving:
                    self._log.append(
                        f"[CRITICAL] {artifact}#{meta.index}: ALL replicas lost!"
                    )
                    continue

                # Wybierz tyle nowych węzłów, ile straciliśmy
                exclude = set(meta.locations)  # nie kopiuj na ten, który już ma
                try:
                    new_nodes = self._pick_nodes(len(lost), exclude=exclude)
                except RuntimeError:
                    self._log.append(
                        f"[CRITICAL] {artifact}#{meta.index}: "
                        f"no replacement nodes available"
                    )
                    continue

                # Pobierz payload z żywej kopii
                src_handle = self.nodes[surviving[0]]["handle"]
                try:
                    payload = await src_handle.fetch.remote(meta.chunk_id)
                except Exception as e:
                    self._log.append(
                        f"[re-repl-error] fetch {meta.chunk_id} from {surviving[0]}: {e}"
                    )
                    continue

                # Wgraj na nowe węzły równolegle
                store_tasks = [
                    self.nodes[nid]["handle"].store.remote(meta.chunk_id, payload)
                    for nid in new_nodes
                ]
                store_results = await asyncio.gather(
                    *store_tasks, return_exceptions=True
                )

                # Zaktualizuj metadane: usuń martwe lokacje, dodaj nowe
                meta.locations = [n for n in meta.locations if n not in dead]
                for nid, res in zip(new_nodes, store_results):
                    if not isinstance(res, Exception):
                        meta.locations.append(nid)
                self._log.append(
                    f"[re-repl] {artifact}#{meta.index}: "
                    f"lost={lost} -> new={new_nodes}"
                )

## 5. `DFSClient` — wrapper dla użytkownika

Klient jest zwykłą klasą Pythona (nie aktorem). Jego rola:

1. Złapać NameNode po nazwie (`ray.get_actor`).
2. Tnąć artefakty na chunki, liczyć hashe.
3. **Bezpośrednio** pisać chunki na DataNodes po dostaniu planu od NameNode.
4. Po sukcesie wszystkich zapisów — wołać `commit_*` na NameNode.

NameNode nigdy nie widzi payloadów (poza re-replikacją przez fetch między
DataNodes).

In [29]:
class DFSClient:
    def __init__(self, namenode_name: str = "NameNode", namespace: str = "dfs"):
        self.namenode = ray.get_actor(namenode_name, namespace=namespace)
        # Cache uchwytów DataNodes po node_id — uzupełniany leniwie
        self._node_cache: dict = {}

    async def _get_node(self, node_id: str):
        if node_id not in self._node_cache:
            self._node_cache[node_id] = await self.namenode.get_node_handle.remote(node_id)
        return self._node_cache[node_id]

    # ---- upload 
    async def upload(self, name: str, content: str) -> dict:
        chunks = chunk_string(content)
        plan = await self.namenode.plan_placement.remote(name, len(chunks))

        commit_specs = []
        write_tasks = []
        for idx, (payload, targets) in enumerate(zip(chunks, plan)):
            h = chunk_hash(payload)
            chunk_id = f"{name}__{idx:04d}__{h}"
            commit_specs.append({
                "index": idx,
                "chunk_id": chunk_id,
                "hash": h,
                "locations": targets,
            })
            for nid in targets:
                handle = await self._get_node(nid)
                write_tasks.append(handle.store.remote(chunk_id, payload))

        results = await asyncio.gather(*write_tasks, return_exceptions=True)
        errors = [str(r) for r in results if isinstance(r, Exception)]
        if errors:
            raise RuntimeError(f"Upload failed on some chunks: {errors}")

        await self.namenode.commit_artifact.remote(name, commit_specs)
        return {"artifact": name, "chunks": len(chunks), "size": len(content)}

    # ---- download 
    async def download(self, name: str) -> str:
        metas = await self.namenode.get_artifact_metadata.remote(name)

        async def fetch_chunk(meta):
            # Spróbuj każdej lokacji po kolei — jak jedna padła, weź następną
            last_err = None
            for nid in meta["locations"]:
                try:
                    handle = await self._get_node(nid)
                    return meta["index"], await handle.fetch.remote(meta["chunk_id"])
                except Exception as e:
                    last_err = e
                    continue
            raise RuntimeError(
                f"All replicas unreachable for chunk {meta['chunk_id']}: {last_err}"
            )

        results = await asyncio.gather(*[fetch_chunk(m) for m in metas])
        # Posortuj po indeksie i sklej
        results.sort(key=lambda x: x[0])
        return "".join(payload for _, payload in results)

    # ---- update (inkrementalny)
    async def update(self, name: str, new_content: str) -> dict:
        new_chunks = chunk_string(new_content)
        chunk_meta = [(payload, chunk_hash(payload)) for payload in new_chunks]
        plan = await self.namenode.plan_update.remote(name, chunk_meta)

        # Zaaplikuj zapisy zmienionych pozycji
        write_tasks = []
        for spec in plan["to_write"]:
            chunk_id = f"{name}__{spec['index']:04d}__{spec['hash']}"
            spec["chunk_id"] = chunk_id  # przekazujemy z powrotem do commitu
            for nid in spec["targets"]:
                handle = await self._get_node(nid)
                write_tasks.append(handle.store.remote(chunk_id, spec["payload"]))

        results = await asyncio.gather(*write_tasks, return_exceptions=True)
        errors = [str(r) for r in results if isinstance(r, Exception)]
        if errors:
            raise RuntimeError(f"Update writes failed: {errors}")

        # Usuń odpadłe chunki
        delete_tasks = []
        for spec in plan["to_delete"]:
            for nid in spec["locations"]:
                handle = await self._get_node(nid)
                delete_tasks.append(handle.remove.remote(spec["chunk_id"]))
        await asyncio.gather(*delete_tasks, return_exceptions=True)

        # Złóż nową listę metadanych: unchanged + nowe zapisane
        old_metas = await self.namenode.get_artifact_metadata.remote(name)
        old_by_idx = {m["index"]: m for m in old_metas}
        new_metas = []
        for idx in plan["unchanged"]:
            new_metas.append(old_by_idx[idx])
        for spec in plan["to_write"]:
            new_metas.append({
                "index": spec["index"],
                "chunk_id": spec["chunk_id"],
                "hash": spec["hash"],
                "locations": spec["targets"],
            })

        await self.namenode.commit_update.remote(name, new_metas)
        return {
            "artifact": name,
            "reused_chunks": len(plan["unchanged"]),
            "rewritten_chunks": len(plan["to_write"]),
            "dropped_chunks": len(plan["to_delete"]),
        }

    # ---- delete
    async def delete(self, name: str) -> dict:
        return await self.namenode.delete_artifact.remote(name)

    # ---- list / status 
    async def list_artifacts(self) -> list[str]:
        return await self.namenode.list_artifacts.remote()

    async def status(self) -> dict:
        return await self.namenode.cluster_status.remote()

## 6. Bootstrap klastra DFS

Tworzymy NameNode (named, detached) i 5 DataNodes. NameNode rejestruje wszystkie
DataNodes i startuje pętlę heartbeat.

In [30]:
NUM_DATA_NODES = 5

# Detached named actor — przeżyje skrypt i będzie dostępny po imieniu
namenode = NameNode.options(
    name="NameNode",
    namespace="dfs",
    lifetime="detached",
).remote(replication_factor=REPLICATION_FACTOR)

data_nodes = {}
for i in range(NUM_DATA_NODES):
    node_id = f"dn-{i+1}"
    handle = DataNode.options(name=node_id, namespace="dfs", lifetime="detached").remote(node_id)
    data_nodes[node_id] = handle
    ray.get(namenode.register_node.remote(node_id, handle))

ray.get(namenode.start_monitoring.remote())
print(f"Cluster up: 1 NameNode + {len(data_nodes)} DataNodes, RF={REPLICATION_FACTOR}")

Cluster up: 1 NameNode + 5 DataNodes, RF=3


## 7. Helper do ładnego wyświetlania statusu

In [31]:
def print_status(status: dict, title: str = ""):
    if title:
        print(f"\n=== {title} ===")
    print(f"Replication factor: {status['replication_factor']}")
    print("\nNodes:")
    for nid, info in sorted(status["nodes"].items()):
        if info.get("status") == "UNREACHABLE":
            print(f"  {nid}: UNREACHABLE ({info.get('error', '')[:60]})")
            continue
        nv = info.get("namenode_view", "?")
        own = info.get("status", "?")
        cnt = info.get("chunk_count", 0)
        marker = "Pass" if nv == "READY" else "Fail"
        print(f"  {marker} {nid}: NN-view={nv:5s}  self={own:5s}  chunks={cnt}")
    print("\nArtifacts:")
    if not status["artifacts"]:
        print("  (none)")
    for name, chunks in status["artifacts"].items():
        print(f"  {name}: {len(chunks)} chunk(s)")
        for c in chunks:
            locs = ",".join(c["locations"])
            print(f"    #{c['index']:02d} hash={c['hash']} @ [{locs}]")


async def show(title: str = ""):
    client = DFSClient()
    s = await client.status()
    print_status(s, title)

## 8. Demo + testy — pełen scenariusz

1. **Upload** dwóch artefaktów - sprawdzamy rozkład chunków.
2. **List** — listujemy artefakty i status węzłów.
3. **Download** — pobieramy artefakt i porównujemy z oryginałem (asercja).
4. **Update inkrementalny** — zmieniamy **jeden znak** w środku artefaktu i sprawdzamy,
   że tylko jeden chunk został przepisany (`reused_chunks` >> `rewritten_chunks`).
5. **Awaria DataNode** — zabijamy jeden węzeł, czekamy aż heartbeat go wykryje,
   i sprawdzamy, że NameNode re-replikował chunki na żywe węzły.
6. **Spójność po awarii** — wszystkie kopie chunku mają identyczną zawartość.
7. **Delete** — usuwamy artefakt i sprawdzamy, że chunki znikają z DataNodes.

In [ ]:
async def demo():
    client = DFSClient()

    # --- 1. Upload 
    print("\n" + "#" * 60)
    print("# 1. UPLOAD")
    print("#" * 60)

    art1 = (
        "Lorem ipsum dolor sit amet, consectetur adipiscing elit. "
        "Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. "
        "Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris."
    )
    art2 = "x" * 200  # 200 znaków same iksy → 7 chunków po 32 (ostatni krótszy)

    r1 = await client.upload("lorem", art1)
    r2 = await client.upload("xfile", art2)
    print(f"  uploaded lorem: {r1}")
    print(f"  uploaded xfile: {r2}")
    await show("PO UPLOADZIE")

    # --- 2. List
    print("\n" + "#" * 60)
    print("# 2. LIST")
    print("#" * 60)
    arts = await client.list_artifacts()
    print(f"  artifacts in cluster: {arts}")
    assert set(arts) == {"lorem", "xfile"}, "list_artifacts mismatch"

    # --- 3. Download and integrity
    print("\n" + "#" * 60)
    print("# 3. DOWNLOAD + INTEGRITY CHECK")
    print("#" * 60)
    dl1 = await client.download("lorem")
    dl2 = await client.download("xfile")
    assert dl1 == art1, "lorem mismatch!"
    assert dl2 == art2, "xfile mismatch!"
    print(f"   lorem matches ({len(dl1)} chars)")
    print(f"   xfile matches ({len(dl2)} chars)")

    # --- 4. Update inkrementalny
    print("\n" + "#" * 60)
    print("# 4. INCREMENTAL UPDATE — zmieniamy 1 znak w środku artefaktu")
    print("#" * 60)
    # Zmień jeden znak na pozycji 100 (zmieni tylko ~1 chunk z ~6)
    art1_modified = art1[:100] + "Z" + art1[101:]
    diffs_with_original = sum(1 for a, b in zip(art1, art1_modified) if a != b)
    print(f"  characters changed: {diffs_with_original}")
    upd = await client.update("lorem", art1_modified)
    print(f"  update result: {upd}")
    assert upd["rewritten_chunks"] == 1, \
        f"expected 1 chunk rewritten, got {upd['rewritten_chunks']}"
    assert upd["reused_chunks"] >= 5, \
        f"expected most chunks reused, got {upd['reused_chunks']}"
    print(f"   inkrementalność potwierdzona: "
          f"{upd['reused_chunks']} chunków zachowanych, tylko {upd['rewritten_chunks']} przepisany")

    # Pobierz i porównaj
    dl_after = await client.download("lorem")
    assert dl_after == art1_modified, "lorem mismatch after update!"
    print(f"   download po update zgadza się z nową wersją")
    await show("PO INKREMENTALNYM UPDATE")

    # --- 5. Awaria DataNode + re-replikacja 
    print("\n" + "#" * 60)
    print("# 5. AWARIA DATANODE")
    print("#" * 60)

    victim = "dn-2"
    print(f"  killing {victim} ...")
    await data_nodes[victim].kill.remote()

    print(f"  waiting for heartbeat to detect & re-replicate (~5s)...")
    await asyncio.sleep(5.0)
    await show("PO AWARII + RE-REPLIKACJI")

    # Asercja: żaden artefakt nie powinien mieć chunka z `dn-2` w lokacjach
    status_after = await client.status()
    leaked = []
    for art_name, chunks in status_after["artifacts"].items():
        for c in chunks:
            if victim in c["locations"]:
                leaked.append((art_name, c["index"]))
    assert not leaked, f"Metadata still references dead node: {leaked}"
    print(f"   metadata clean — no references to {victim}")

    # Każdy chunk nadal ma RF=3 lokacji
    for art_name, chunks in status_after["artifacts"].items():
        for c in chunks:
            assert len(c["locations"]) == REPLICATION_FACTOR, \
                f"{art_name}#{c['index']} has {len(c['locations'])} replicas"
    print(f"   all chunks restored to RF={REPLICATION_FACTOR}")

    # Download po awarii nadal działa
    dl_recovered = await client.download("lorem")
    assert dl_recovered == art1_modified, "lorem corrupted after failure!"
    print(f"   download po awarii działa — dane spójne")

    # --- 6. Spójność wszystkich kopii
    print("\n" + "#" * 60)
    print("# 6. CONSISTENCY CHECK — wszystkie kopie chunku identyczne")
    print("#" * 60)
    for art_name, chunks in status_after["artifacts"].items():
        for c in chunks:
            payloads = await asyncio.gather(*[
                data_nodes[nid].fetch.remote(c["chunk_id"])
                for nid in c["locations"]
            ])
            assert len(set(payloads)) == 1, \
                f"{art_name}#{c['index']}: replicas differ! {payloads}"
    print(f"   all replicas of all chunks are bit-identical")

    # --- 7. Delete --------------------------------------------------------
    print("\n" + "#" * 60)
    print("# 7. DELETE")
    print("#" * 60)
    chunks_before = sum(
        info["chunk_count"] for info in status_after["nodes"].values()
        if isinstance(info.get("chunk_count"), int)
    )
    del_result = await client.delete("xfile")
    print(f"  delete result: {del_result}")
    status_final = await client.status()
    chunks_after = sum(
        info["chunk_count"] for info in status_final["nodes"].values()
        if isinstance(info.get("chunk_count"), int)
    )
    # xfile miał 200/32 = 7 chunków × RF=3 = 21 fizycznych kopii
    print(f"  physical chunks: {chunks_before} -> {chunks_after} "
          f"(diff={chunks_before - chunks_after})")
    assert "xfile" not in (await client.list_artifacts())
    print(f"   xfile removed from metadata")

    await show("STAN KOŃCOWY")

    print("\n" + "=" * 60)
    print("ALL TESTS PASSED")
    print("=" * 60)


await demo()


############################################################
# 1. UPLOAD
############################################################
  uploaded lorem: {'artifact': 'lorem', 'chunks': 6, 'size': 191}
  uploaded xfile: {'artifact': 'xfile', 'chunks': 7, 'size': 200}

=== PO UPLOADZIE ===
Replication factor: 3

Nodes:
  Pass dn-1: NN-view=READY  self=READY  chunks=7
  Pass dn-2: NN-view=READY  self=READY  chunks=8
  Pass dn-3: NN-view=READY  self=READY  chunks=9
  Pass dn-4: NN-view=READY  self=READY  chunks=8
  Pass dn-5: NN-view=READY  self=READY  chunks=7

Artifacts:
  lorem: 6 chunk(s)
    #00 hash=d6fbb2434692 @ [dn-1,dn-2,dn-3]
    #01 hash=1629dbfd6228 @ [dn-2,dn-3,dn-4]
    #02 hash=373aa7930e32 @ [dn-3,dn-4,dn-5]
    #03 hash=ff76637da5e9 @ [dn-4,dn-5,dn-1]
    #04 hash=89b65e44968e @ [dn-5,dn-1,dn-2]
    #05 hash=1f70582d29da @ [dn-1,dn-2,dn-3]
  xfile: 7 chunk(s)
    #00 hash=dc8fe1d6497e @ [dn-2,dn-3,dn-4]
    #01 hash=dc8fe1d6497e @ [dn-3,dn-4,dn-5]
    #02 hash=dc8fe1d6497

## 9. Log zdarzeń NameNode

Pełen ślad operacji — przydatny do prezentacji na obronie.

In [33]:
client = DFSClient()
log = ray.get(namenode.get_log.remote())
for line in log:
    print(line)

[register] dn-1
[register] dn-2
[register] dn-3
[register] dn-4
[register] dn-5
[monitor] heartbeat loop started
[commit] lorem: 6 chunks
[commit] xfile: 7 chunks
[update-plan] lorem: unchanged=5 rewrite=1 drop=0
[update] lorem: now 6 chunks
[health] dn-2: READY -> DOWN
[re-repl] lorem#0: lost=['dn-2'] -> new=['dn-4']
[re-repl] lorem#1: lost=['dn-2'] -> new=['dn-5']
[re-repl] lorem#4: lost=['dn-2'] -> new=['dn-3']
[re-repl] lorem#5: lost=['dn-2'] -> new=['dn-5']
[re-repl] xfile#0: lost=['dn-2'] -> new=['dn-1']
[re-repl] xfile#3: lost=['dn-2'] -> new=['dn-4']
[re-repl] xfile#4: lost=['dn-2'] -> new=['dn-4']
[re-repl] xfile#5: lost=['dn-2'] -> new=['dn-5']
[delete] xfile: removed 7 chunk(s)


## 10. Cleanup

Detached actors trzeba ubić jawnie — inaczej zostają w klastrze.

In [34]:
ray.get(namenode.stop_monitoring.remote())
for nid, handle in data_nodes.items():
    ray.kill(handle)
ray.kill(namenode)
ray.shutdown()
print("Cluster down.")

Cluster down.
